# 🥔 Potato Disease Classification - Final Training

## Model: DenseNet121 (Best Performer - 100% Accuracy! 🎉)

### Features:
- ✅ Two-phase training (Feature Extraction → Fine-tuning)
- ✅ Extended epochs for maximum accuracy
- ✅ Quantization (28MB → ~7MB)
- ✅ TFLite export for Flutter
- ✅ Comprehensive evaluation and visualization

### Output Files:
- `best_model.h5` - Full Keras model
- `potato_model.tflite` - Standard TFLite
- `potato_model_quantized.tflite` - Quantized TFLite (~7MB)
- `labels.txt` - Class labels
- `model_metadata.json` - Model info
- Training history and evaluation charts

### Comparison Results:
- **DenseNet121**: 100% accuracy, 98.98% confidence, 28.0 MB
- Selected as the best model for final training


In [ ]:
# ========================================
# CELL 1: Mount Drive & Setup
# ========================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted!")


In [ ]:
# ========================================
# CELL 2: Imports and Configuration
# ========================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ========================================
# CONFIGURATION
# ========================================

# Paths (using organized folder from comparison notebook)
DATASET_PATH = '/content/drive/MyDrive/potato_dataset/organized'
MODEL_PATH = '/content/drive/MyDrive/models'
os.makedirs(MODEL_PATH, exist_ok=True)

# Training parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
PHASE1_EPOCHS = 15      # Feature extraction (extended)
PHASE2_EPOCHS = 30      # Fine-tuning (extended for maximum accuracy)
PHASE1_LR = 0.001
PHASE2_LR = 0.00005     # Lower LR for fine-tuning
UNFREEZE_LAYERS = 50    # Unfreeze more layers for better performance

# Class names
CLASS_NAMES = ['late_blight', 'healthy', 'bacterial_wilt', 'not_potato']
NUM_CLASSES = 4

print("🥔 POTATO DISEASE CLASSIFIER - FINAL TRAINING")
print("=" * 60)
print(f"Model: DenseNet121 (100% accuracy in comparison!)")
print(f"Dataset: {DATASET_PATH}")
print(f"Output: {MODEL_PATH}")
print(f"Phase 1: {PHASE1_EPOCHS} epochs (frozen base)")
print(f"Phase 2: {PHASE2_EPOCHS} epochs (fine-tuning)")
print(f"Classes: {CLASS_NAMES}")

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"\n✅ GPU available: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("\n⚠️ No GPU - training will be slower")


In [ ]:
# ========================================
# CELL 3: Create Data Generators
# ========================================

print("📊 Creating Data Generators...")

# Light augmentation (balanced dataset)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    zoom_range=0.1
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'train'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=True,
    seed=42
)

val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'val'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'test'),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

print(f"\n✅ Generators Created:")
print(f"   Train: {train_generator.samples} samples, {len(train_generator)} batches")
print(f"   Val: {val_generator.samples} samples, {len(val_generator)} batches")
print(f"   Test: {test_generator.samples} samples, {len(test_generator)} batches")
print(f"\n   Class indices: {train_generator.class_indices}")

# Compute class weights (should be balanced, but compute anyway)
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = dict(enumerate(class_weights))

print("\n⚖️ Class Weights:")
for name, idx in train_generator.class_indices.items():
    print(f"   {name}: {class_weight_dict[idx]:.3f}")


In [ ]:
# ========================================
# CELL 4: Build Model
# ========================================

print("🏗️ Building DenseNet121 model...")

# Base model
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False

# Build model
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

model = Model(inputs, outputs, name='potato_densenet121')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE1_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"✅ Model built!")
print(f"   Total params: {model.count_params():,}")
model.summary()


In [ ]:
# ========================================
# CELL 5: Phase 1 - Feature Extraction
# ========================================

print("\n" + "=" * 60)
print("📌 PHASE 1: FEATURE EXTRACTION")
print("=" * 60)

# Callbacks
checkpoint_path = os.path.join(MODEL_PATH, 'phase1_best.h5')
callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-7,
        verbose=1
    ),
    CSVLogger(os.path.join(MODEL_PATH, 'phase1_history.csv'))
]

# Train
print(f"\nTraining for {PHASE1_EPOCHS} epochs with frozen base...")
history1 = model.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 1 complete!")


In [ ]:
# ========================================
# CELL 6: Phase 2 - Fine-tuning
# ========================================

print("\n" + "=" * 60)
print("📌 PHASE 2: FINE-TUNING")
print("=" * 60)

# Unfreeze top layers
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=PHASE2_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_count = sum([tf.reduce_prod(v.shape).numpy() for v in model.trainable_variables])
total_count = sum([tf.reduce_prod(v.shape).numpy() for v in model.variables])
print(f"\n   Trainable params: {trainable_count:,} / {total_count:,} ({100*trainable_count/total_count:.1f}%)")

# Callbacks
checkpoint_path = os.path.join(MODEL_PATH, 'best_model.h5')
callbacks = [
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-8,
        verbose=1
    ),
    CSVLogger(os.path.join(MODEL_PATH, 'phase2_history.csv'))
]

# Train
print(f"\nFine-tuning for {PHASE2_EPOCHS} epochs...")
history2 = model.fit(
    train_generator,
    epochs=PHASE2_EPOCHS,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Phase 2 complete!")

# Load best model
best_model = keras.models.load_model(checkpoint_path)
print(f"\n✅ Best model loaded: {checkpoint_path}")


In [ ]:
# ========================================
# CELL 7: Clean Corrupted Images from Test Set
# ========================================

print("🧹 Cleaning corrupted images from test set...")
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

corrupted_count = 0
for class_name in CLASS_NAMES:
    test_class_path = os.path.join(DATASET_PATH, 'test', class_name)
    if not os.path.exists(test_class_path):
        continue
    
    images = [f for f in os.listdir(test_class_path) 
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    for img_file in images:
        img_path = os.path.join(test_class_path, img_file)
        try:
            with Image.open(img_path) as img:
                img.verify()
            img = Image.open(img_path)
            img.load()
            img.close()
        except Exception as e:
            print(f"   🗑️ Removing corrupted: test/{class_name}/{img_file}")
            try:
                os.remove(img_path)
                corrupted_count += 1
            except:
                pass

if corrupted_count > 0:
    print(f"✅ Cleaned {corrupted_count} corrupted images from test set")
    print("   Recreating test generator...")
    # Recreate test generator after cleaning
    test_generator = val_test_datagen.flow_from_directory(
        os.path.join(DATASET_PATH, 'test'),
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES,
        shuffle=False
    )
    print(f"   Test samples after cleaning: {test_generator.samples}")
else:
    print("✅ No corrupted images found in test set")


In [ ]:
# ========================================
# CELL 8: Evaluation
# ========================================

print("\n" + "=" * 60)
print("📊 EVALUATION")
print("=" * 60)

# Evaluate on test set (with error handling)
try:
    test_generator.reset()
    test_loss, test_acc = best_model.evaluate(test_generator, verbose=1)
except Exception as e:
    print(f"\n⚠️ Error during evaluation: {str(e)}")
    print("   Attempting to evaluate with error handling...")
    
    # Manual evaluation with error handling
    test_generator.reset()
    correct = 0
    total = 0
    all_predictions = []
    all_true_classes = []
    
    for batch_idx in range(len(test_generator)):
        try:
            batch_x, batch_y = test_generator[batch_idx]
            batch_pred = best_model.predict(batch_x, verbose=0)
            batch_pred_classes = np.argmax(batch_pred, axis=1)
            batch_true_classes = np.argmax(batch_y, axis=1)
            
            correct += np.sum(batch_pred_classes == batch_true_classes)
            total += len(batch_true_classes)
            all_predictions.extend(batch_pred_classes)
            all_true_classes.extend(batch_true_classes)
        except Exception as batch_error:
            print(f"   ⚠️ Skipping batch {batch_idx}: {str(batch_error)}")
            continue
    
    test_acc = correct / total if total > 0 else 0.0
    test_loss = 0.0  # Can't compute loss without full batch
    predicted_classes = np.array(all_predictions)
    true_classes = np.array(all_true_classes)
    
    print(f"\n✅ Manual evaluation complete: {correct}/{total} correct")

print(f"\n✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
if test_loss > 0:
    print(f"✅ Test Loss: {test_loss:.4f}")

# Get predictions (if not already computed)
if 'predictions' not in locals():
    print("\n📊 Getting predictions...")
    test_generator.reset()
    predictions_list = []
    true_classes_list = []
    
    for batch_idx in range(len(test_generator)):
        try:
            batch_x, batch_y = test_generator[batch_idx]
            batch_pred = best_model.predict(batch_x, verbose=0)
            batch_true = np.argmax(batch_y, axis=1)
            
            predictions_list.append(batch_pred)
            true_classes_list.extend(batch_true)
        except Exception as e:
            print(f"   ⚠️ Skipping batch {batch_idx}: {str(e)}")
            continue
    
    predictions = np.vstack(predictions_list)
    true_classes = np.array(true_classes_list)
    predicted_classes = np.argmax(predictions, axis=1)
else:
    # Use already computed values
    true_classes = test_generator.classes
    predicted_classes = np.argmax(predictions, axis=1)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(
    true_classes,
    predicted_classes,
    target_names=CLASS_NAMES
))

# Confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix\nTest Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)',
          fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_PATH, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print("\n📊 Per-Class Accuracy:")
for i, class_name in enumerate(CLASS_NAMES):
    status = "✅" if per_class_acc[i] >= 0.95 else "⚠️" if per_class_acc[i] >= 0.85 else "❌"
    print(f"   {status} {class_name}: {per_class_acc[i]:.4f} ({per_class_acc[i]*100:.2f}%)")

# Average confidence
avg_confidence = np.mean(np.max(predictions, axis=1))
print(f"\n✅ Average Confidence: {avg_confidence:.4f} ({avg_confidence*100:.2f}%)")


In [ ]:
# ========================================
# CELL 9: Training History Visualization
# ========================================

print("\n📈 Plotting training history...")

# Combine histories
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history.keys():
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined_history = combine_histories(history1, history2)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(combined_history['accuracy'], label='Train', linewidth=2)
axes[0].plot(combined_history['val_accuracy'], label='Val', linewidth=2)
axes[0].axvline(len(history1.history['accuracy']), color='r', linestyle='--', label='Phase 2 Start')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(combined_history['loss'], label='Train', linewidth=2)
axes[1].plot(combined_history['val_loss'], label='Val', linewidth=2)
axes[1].axvline(len(history1.history['loss']), color='r', linestyle='--', label='Phase 2 Start')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_PATH, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history saved!")


In [ ]:
# ========================================
# CELL 10: Save Labels
# ========================================

labels_path = os.path.join(MODEL_PATH, 'labels.txt')
with open(labels_path, 'w') as f:
    for class_name in CLASS_NAMES:
        f.write(f"{class_name}\n")

print(f"✅ Labels saved: {labels_path}")
print(f"   Classes: {', '.join(CLASS_NAMES)}")


In [ ]:
# ========================================
# CELL 11: Export to TFLite (Standard)
# ========================================

print("\n" + "=" * 60)
print("📱 EXPORTING TO TFLITE")
print("=" * 60)

# Standard TFLite
print("\n1️⃣ Converting to standard TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

tflite_path = os.path.join(MODEL_PATH, 'potato_model.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"   ✅ Standard TFLite saved: {tflite_path}")
print(f"   Size: {size_mb:.2f} MB")


In [ ]:
# ========================================
# CELL 13: Verify TFLite Model Accuracy
# ========================================

print("\n" + "=" * 60)
print("🔬 VERIFYING TFLITE MODELS")
print("=" * 60)

def evaluate_tflite_model(tflite_path, test_generator, class_names):
    """Evaluate TFLite model accuracy on test set."""
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    correct = 0
    total = 0
    confidences = []
    
    test_generator.reset()
    
    for batch_x, batch_y in test_generator:
        for i in range(batch_x.shape[0]):
            # Prepare input
            input_data = np.expand_dims(batch_x[i], axis=0).astype(np.float32)
            interpreter.set_tensor(input_details[0]['index'], input_data)
            interpreter.invoke()
            
            # Get output
            output_data = interpreter.get_tensor(output_details[0]['index'])
            pred_class = np.argmax(output_data[0])
            true_class = np.argmax(batch_y[i])
            confidence = np.max(output_data[0])
            
            if pred_class == true_class:
                correct += 1
            total += 1
            confidences.append(confidence)
        
        if total >= test_generator.samples:
            break
    
    accuracy = correct / total
    avg_confidence = np.mean(confidences)
    return accuracy, avg_confidence

# Evaluate both models
print("\n📊 Evaluating standard TFLite model...")
std_acc, std_conf = evaluate_tflite_model(tflite_path, test_generator, CLASS_NAMES)
print(f"   Accuracy: {std_acc:.4f} ({std_acc*100:.2f}%)")
print(f"   Avg Confidence: {std_conf:.4f} ({std_conf*100:.2f}%)")

print("\n📊 Evaluating quantized TFLite model...")
quant_acc, quant_conf = evaluate_tflite_model(quantized_path, test_generator, CLASS_NAMES)
print(f"   Accuracy: {quant_acc:.4f} ({quant_acc*100:.2f}%)")
print(f"   Avg Confidence: {quant_conf:.4f} ({quant_conf*100:.2f}%)")

# Comparison
acc_drop = (std_acc - quant_acc) * 100
conf_drop = (std_conf - quant_conf) * 100

print(f"\n📈 Comparison:")
print(f"   Accuracy Drop: {acc_drop:.2f}%")
print(f"   Confidence Drop: {conf_drop:.2f}%")

if acc_drop < 1 and conf_drop < 2:
    print("\n   ✅ Excellent! Quantized model performs well - RECOMMENDED for Flutter app")
elif acc_drop < 3 and conf_drop < 5:
    print("\n   ⚠️ Moderate accuracy/confidence loss - Consider using standard model")
    print("   💡 You can use potato_model.tflite (standard) if accuracy is critical")
else:
    print("\n   ❌ Significant accuracy loss - USE STANDARD MODEL INSTEAD")
    print("   💡 Use potato_model.tflite (standard) for better accuracy")
    std_size = os.path.getsize(tflite_path) / (1024 * 1024)
    print(f"   📱 Standard model size: {std_size:.2f} MB (still acceptable for mobile)")



In [ ]:
# ========================================
# CELL 12: Export to TFLite (Quantized)
# ========================================

print("\n2️⃣ Converting to quantized TFLite (Dynamic Range Quantization)...")

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_tflite_model = converter.convert()

quantized_path = os.path.join(MODEL_PATH, 'potato_model_quantized.tflite')
with open(quantized_path, 'wb') as f:
    f.write(quantized_tflite_model)

size_mb = os.path.getsize(quantized_path) / (1024 * 1024)
std_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
reduction = (1 - size_mb/std_size_mb) * 100

print(f"   ✅ Quantized TFLite saved: {quantized_path}")
print(f"   Size: {size_mb:.2f} MB")
print(f"   Reduction: {reduction:.1f}% (from {std_size_mb:.2f} MB)")

# Verify quantization
print("\n📊 Model Info:")
print(f"   Standard: {std_size_mb:.2f} MB")
print(f"   Quantized: {size_mb:.2f} MB")
print(f"   Input: {IMG_SIZE[0]}x{IMG_SIZE[1]} RGB")
print(f"   Output: 4-class softmax probabilities")
print(f"\n✅ Use quantized model for Flutter app (smaller size, minimal accuracy loss)")


In [ ]:
# ========================================
# CELL 13: Save Model Metadata
# ========================================

metadata = {
    'model_name': 'Potato Disease Classifier',
    'architecture': 'DenseNet121',
    'version': '1.0',
    'created_at': datetime.now().isoformat(),
    'classes': CLASS_NAMES,
    'num_classes': NUM_CLASSES,
    'input_size': IMG_SIZE,
    'test_accuracy': float(test_acc),
    'average_confidence': float(avg_confidence),
    'per_class_accuracy': {CLASS_NAMES[i]: float(per_class_acc[i]) for i in range(len(CLASS_NAMES))},
    'training_params': {
        'phase1_epochs': PHASE1_EPOCHS,
        'phase2_epochs': PHASE2_EPOCHS,
        'batch_size': BATCH_SIZE,
        'phase1_lr': PHASE1_LR,
        'phase2_lr': PHASE2_LR
    },
    'files': {
        'keras_model': 'best_model.h5',
        'tflite_standard': 'potato_model.tflite',
        'tflite_quantized': 'potato_model_quantized.tflite',
        'labels': 'labels.txt'
    }
}

metadata_path = os.path.join(MODEL_PATH, 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Metadata saved: {metadata_path}")


In [ ]:
# ========================================
# CELL 14: Final Summary
# ========================================

print("\n" + "=" * 70)
print("🎉 TRAINING COMPLETE!")
print("=" * 70)

print(f"""
📁 OUTPUT FILES IN: {MODEL_PATH}

┌─────────────────────────────────────────────────────────────────────┐
│ FILE                           │ SIZE     │ PURPOSE                │
├─────────────────────────────────────────────────────────────────────┤
│ best_model.h5                  │ ~28 MB   │ Full Keras model       │
│ potato_model.tflite            │ ~28 MB   │ Standard TFLite        │
│ potato_model_quantized.tflite │ ~7 MB    │ Quantized (for mobile) │
│ labels.txt                     │ <1 KB    │ Class labels           │
│ model_metadata.json            │ <1 KB    │ Model info             │
│ confusion_matrix.png           │ ~100 KB  │ Evaluation results     │
│ training_history.png           │ ~150 KB  │ Training curves        │
└─────────────────────────────────────────────────────────────────────┘

🎯 RESULTS:
   Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)
   Average Confidence: {avg_confidence:.4f} ({avg_confidence*100:.2f}%)

📊 PER-CLASS ACCURACY:
""")

for i, class_name in enumerate(CLASS_NAMES):
    print(f"   {class_name}: {per_class_acc[i]:.4f} ({per_class_acc[i]*100:.2f}%)")

print(f"""
📱 FOR FLUTTER:
   1. Use: potato_model_quantized.tflite (~7 MB)
   2. Labels: labels.txt
   3. Input: 224x224 RGB image (normalized 0-1)
   4. Output: 4-class softmax probabilities

✅ Ready for deployment!
""")

print("=" * 70)
